# Tabela de Desempenho para Apresentação — ENEM 2022

Este notebook calcula os valores da tabela comparativa de grupos socioeconômicos
a partir dos CSVs de analytics já gerados pelo pipeline do dashboard.

---

## Definição da métrica "Média Geral"

Conforme `notebooks/02_transformação_dados.ipynb` (célula de criação de variáveis):

```python
MEDIA_GERAL = mean([NU_NOTA_CN, NU_NOTA_CH, NU_NOTA_LC, NU_NOTA_MT, NU_NOTA_REDACAO])
```

É a **média aritmética das 5 provas** (4 objetivas + Redação).  
Candidatos com nota parcialmente nula têm a média calculada sobre as notas disponíveis.

---

## Mapeamento dos grupos

| Grupo | Coluna no dataset | Valor filtrado |
|---|---|---|
| Alta Renda (Classe Q) | `RENDA_FAMILIAR` | `"Acima de 20 salários"` (faixa Q do Q006) |
| Escola Privada | `TIPO_ESCOLA` | `"Privada"` (TP_ESCOLA = 3) |
| Com Internet em Casa | `ACESSO_INTERNET` | `"Sim"` (Q025 = B) |
| Média Nacional | — | todos os participantes |

In [41]:
from pathlib import Path
import pandas as pd

PASTA_ANALYTICS = Path.cwd().resolve().parent / "data" / "analytics"

df_resumo   = pd.read_csv(PASTA_ANALYTICS / "resumo_geral_2022.csv",        sep=";", encoding="utf-8")
df_renda    = pd.read_csv(PASTA_ANALYTICS / "agregado_renda_2022.csv",       sep=";", encoding="utf-8")
df_internet = pd.read_csv(PASTA_ANALYTICS / "agregado_internet_2022.csv",    sep=";", encoding="utf-8")
df_escola   = pd.read_csv(PASTA_ANALYTICS / "agregado_tipo_escola_2022.csv", sep=";", encoding="utf-8")

print("Arquivos carregados com sucesso.")

Arquivos carregados com sucesso.


## 1 — Média Nacional (referência)

In [42]:
media_nacional = df_resumo.iloc[0]["MEDIA_GERAL"]
total_part     = int(df_resumo.iloc[0]["TOTAL_PARTICIPANTES"])

print(f"Média Nacional : {media_nacional:.4f} pts")
print(f"Total de participantes: {total_part:,}".replace(",", "."))

Média Nacional : 538.9622 pts
Total de participantes: 2.504.014


## 2 — Alta Renda (faixa Q = "Acima de 20 salários")

In [43]:
# Confirma o rótulo exato da faixa Q no dataset
print("Faixas disponíveis em RENDA_FAMILIAR:")
print(df_renda["RENDA_FAMILIAR"].tolist())

Faixas disponíveis em RENDA_FAMILIAR:
['Nenhuma renda', 'Até 1 salário mínimo', '1 a 1,5 salários', '1,5 a 2 salários', '2 a 2,5 salários', '2,5 a 3 salários', '3 a 4 salários', '4 a 5 salários', '5 a 6 salários', '6 a 7 salários', '7 a 8 salários', '8 a 9 salários', '9 a 10 salários', '10 a 12 salários', '12 a 15 salários', '15 a 20 salários', 'Acima de 20 salários']


In [44]:
linha_q = df_renda[df_renda["RENDA_FAMILIAR"] == "Acima de 20 salários"]

media_alta_renda  = linha_q["MEDIA_GERAL"].iloc[0]
total_alta_renda  = int(linha_q["TOTAL_PARTICIPANTES"].iloc[0])
dif_alta_renda    = (media_alta_renda - media_nacional) / media_nacional * 100

print(f"Média Alta Renda (faixa Q) : {media_alta_renda:.4f} pts")
print(f"Participantes              : {total_alta_renda:,}".replace(",", "."))
print(f"Diferença relativa vs. nacional: {dif_alta_renda:+.2f}%")

Média Alta Renda (faixa Q) : 635.3144 pts
Participantes              : 41.825
Diferença relativa vs. nacional: +17.88%


## 3 — Escola Privada

In [45]:
# Confirma os rótulos disponíveis
print("Tipos de escola disponíveis:")
print(df_escola[["TIPO_ESCOLA", "TOTAL_PARTICIPANTES", "MEDIA_GERAL"]].to_string(index=False))

Tipos de escola disponíveis:
  TIPO_ESCOLA  TOTAL_PARTICIPANTES  MEDIA_GERAL
      Privada               201037   606.711452
Não respondeu              1490185   543.701272
      Pública               812792   513.516246


In [46]:
linha_priv = df_escola[df_escola["TIPO_ESCOLA"] == "Privada"]

media_privada  = linha_priv["MEDIA_GERAL"].iloc[0]
total_privada  = int(linha_priv["TOTAL_PARTICIPANTES"].iloc[0])
dif_privada    = (media_privada - media_nacional) / media_nacional * 100

print(f"Média Escola Privada        : {media_privada:.4f} pts")
print(f"Participantes               : {total_privada:,}".replace(",", "."))
print(f"Diferença relativa vs. nacional: {dif_privada:+.2f}%")

Média Escola Privada        : 606.7115 pts
Participantes               : 201.037
Diferença relativa vs. nacional: +12.57%


## 4 — Com Internet em Casa

In [47]:
# Confirma os rótulos disponíveis
print("Grupos de acesso à internet:")
print(df_internet[["ACESSO_INTERNET", "TOTAL_PARTICIPANTES", "MEDIA_GERAL"]].to_string(index=False))

Grupos de acesso à internet:
ACESSO_INTERNET  TOTAL_PARTICIPANTES  MEDIA_GERAL
            Sim              2298919   544.029594
            Não               205095   482.161222


In [48]:
linha_net = df_internet[df_internet["ACESSO_INTERNET"] == "Sim"]

media_internet  = linha_net["MEDIA_GERAL"].iloc[0]
total_internet  = int(linha_net["TOTAL_PARTICIPANTES"].iloc[0])
dif_internet    = (media_internet - media_nacional) / media_nacional * 100

print(f"Média Com Internet          : {media_internet:.4f} pts")
print(f"Participantes               : {total_internet:,}".replace(",", "."))
print(f"Diferença relativa vs. nacional: {dif_internet:+.2f}%")

Média Com Internet          : 544.0296 pts
Participantes               : 2.298.919
Diferença relativa vs. nacional: +0.94%


## 5 — Tabela Final

In [49]:
tabela = pd.DataFrame([
    {
        "Categoria"          : "Alta Renda (Classe Q)",
        "Participantes"      : total_alta_renda,
        "Média Geral (pts)"  : round(media_alta_renda, 1),
        "Dif. Relativa (%)"  : f"{dif_alta_renda:+.1f}%",
        "Referência"         : "vs. nacional",
    },
    {
        "Categoria"          : "Escola Privada",
        "Participantes"      : total_privada,
        "Média Geral (pts)"  : round(media_privada, 1),
        "Dif. Relativa (%)"  : f"{dif_privada:+.1f}%",
        "Referência"         : "vs. nacional",
    },
    {
        "Categoria"          : "Com Internet em Casa",
        "Participantes"      : total_internet,
        "Média Geral (pts)"  : round(media_internet, 1),
        "Dif. Relativa (%)"  : f"{dif_internet:+.1f}%",
        "Referência"         : "vs. nacional",
    },
    {
        "Categoria"          : "Média Nacional",
        "Participantes"      : total_part,
        "Média Geral (pts)"  : round(media_nacional, 1),
        "Dif. Relativa (%)"  : "Ref. (0%)",
        "Referência"         : "base toda",
    },
])

display(tabela.style.set_caption("Desempenho ENEM 2022 — grupos socioeconômicos vs. média nacional"))

,Categoria,Participantes,Média Geral (pts),Dif. Relativa (%),Referência
0,Alta Renda (Classe Q),41825,635.300000,+17.9%,vs. nacional
1,Escola Privada,201037,606.700000,+12.6%,vs. nacional
2,Com Internet em Casa,2298919,544.000000,+0.9%,vs. nacional
3,Média Nacional,2504014,539.000000,Ref. (0%),base toda


## Premissas e limitações

1. **Métrica `MEDIA_GERAL`** — média aritmética das 5 provas com `mean(axis=1)`.  
   Se um candidato fez apenas 4 provas, a média é calculada sobre as 4 disponíveis.  
   Candidatos com **todas** as notas nulas foram removidos pelo `data_processor.py` (`dropna(how="all")`).

2. **Faixa Q** — mapeada para `"Acima de 20 salários"` (Q006 = Q no dicionário INEP).  
   Corresponde a renda familiar acima de R$ 19.960,00/mês (≈ 20 salários mínimos de 2022).

3. **Escola Privada** — campo `TIPO_ESCOLA = "Privada"` (TP_ESCOLA = 3).  
   Exclui `"Não respondeu"` (~60% dos candidatos), que têm média similar à nacional.

4. **Com Internet** — `ACESSO_INTERNET = "Sim"` (Q025 = B). Inclui ~91,8% dos candidatos.  
   A diferença pequena (+0,9%) reflete que a quase totalidade já tem acesso.

5. **Ano-base** — microdados ENEM 2022 (N = 2.504.014 participantes).

6. **Independência dos grupos** — os grupos se sobrepõem  
   (ex.: um candidato pode ser de escola privada E ter alta renda). Não são mutuamente exclusivos.

---

## Barreira Digital: Conectividade vs Notas

Mesma métrica `MEDIA_GERAL` (média das 5 provas) e mesmo `df_internet` já carregado acima.  
Coluna: `ACESSO_INTERNET` — valores confirmados na célula 4: `"Sim"` / `"Não"` (mapeados de Q025 B/A).

In [50]:
# Reutiliza df_internet já carregado — sem recarregar nada
# media_internet / total_internet já calculados na seção 4 (grupo "Sim")

linha_sem = df_internet[df_internet["ACESSO_INTERNET"] == "Não"]

media_sem   = linha_sem["MEDIA_GERAL"].iloc[0]
total_sem   = int(linha_sem["TOTAL_PARTICIPANTES"].iloc[0])

# media_com e total_com já existem como media_internet / total_internet
media_com  = media_internet
total_com  = total_internet

penalidade_abs = media_com - media_sem
penalidade_rel = penalidade_abs / media_com * 100
pct_sem        = total_sem / (total_com + total_sem) * 100

print("── Sanity check: tabela df_internet completa ──")
print(df_internet[["ACESSO_INTERNET", "TOTAL_PARTICIPANTES", "MEDIA_GERAL"]].to_string(index=False))

── Sanity check: tabela df_internet completa ──
ACESSO_INTERNET  TOTAL_PARTICIPANTES  MEDIA_GERAL
            Sim              2298919   544.029594
            Não               205095   482.161222


In [51]:
# ── Números prontos para copiar no slide ──────────────────────────────────────

def _br(v): return f"{v:.1f}".replace(".", ",")

print("=" * 50)
print(f"  Com Internet:  {_br(media_com)} pts")
print(f"  Sem Internet:  {_br(media_sem)} pts")
print(f"  Penalidade:    ~{_br(penalidade_abs)} pontos  (−{_br(penalidade_rel)}%)")
print(f"  (n_com = {total_com:,}  |  n_sem = {total_sem:,}  |  % sem = {_br(pct_sem)}%)".replace(",", "."))
print("=" * 50)

# ── Dicionário reutilizável ────────────────────────────────────────────────────
barreira_digital = {
    "media_com_internet"   : round(media_com,       1),
    "media_sem_internet"   : round(media_sem,       1),
    "penalidade_absoluta"  : round(penalidade_abs,  1),
    "penalidade_relativa_pct": round(penalidade_rel, 1),
    "n_com_internet"       : total_com,
    "n_sem_internet"       : total_sem,
    "pct_sem_internet"     : round(pct_sem,         1),
}

print("\nbarreira_digital =", barreira_digital)

  Com Internet:  544,0 pts
  Sem Internet:  482,2 pts
  Penalidade:    ~61,9 pontos  (−11,4%)
  (n_com = 2.298.919  |  n_sem = 205.095  |  % sem = 8.2%)

barreira_digital = {'media_com_internet': np.float64(544.0), 'media_sem_internet': np.float64(482.2), 'penalidade_absoluta': np.float64(61.9), 'penalidade_relativa_pct': np.float64(11.4), 'n_com_internet': 2298919, 'n_sem_internet': 205095, 'pct_sem_internet': 8.2}


In [52]:
import plotly.graph_objects as go
from pathlib import Path

COR_COM = "#18BC9C"   # teal
COR_SEM = "#8FA3B0"   # cinza-azulado
COR_FONTE = "#2C3E50"

grupos  = ["Sem Internet", "Com Internet"]
medias  = [barreira_digital["media_sem_internet"], barreira_digital["media_com_internet"]]
cores   = [COR_SEM, COR_COM]
labels  = [f"{_br(v)} pts" for v in medias]

fig = go.Figure(go.Bar(
    x=medias,
    y=grupos,
    orientation="h",
    marker_color=cores,
    text=labels,
    textposition="outside",
    textfont=dict(size=15, color=COR_FONTE, family="Arial"),
    cliponaxis=False,
))

fig.update_layout(
    xaxis=dict(
        range=[430, 590],
        showgrid=False,
        showticklabels=False,
        zeroline=False,
    ),
    yaxis=dict(
        tickfont=dict(size=14, color=COR_FONTE, family="Arial"),
        showgrid=False,
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=130, r=80, t=60, b=40),
    title=dict(
        text=(
            f"Barreira Digital: acesso à internet × nota ENEM 2022<br>"
            f"<sup>Penalidade de ~{_br(barreira_digital['penalidade_absoluta'])} pontos "
            f"(−{_br(barreira_digital['penalidade_relativa_pct'])}%) "
            f"para os {_br(barreira_digital['pct_sem_internet'])}% sem acesso</sup>"
        ),
        font=dict(size=14, color=COR_FONTE, family="Arial"),
        x=0,
        xanchor="left",
    ),
    width=700,
    height=280,
)

fig.show()

# Exporta PNG para inserir no slide
saida_png = Path.cwd() / "barreira_digital.png"
try:
    fig.write_image(str(saida_png), scale=2)
    print(f"PNG salvo em: {saida_png}")
except Exception as e:
    print(f"Export PNG falhou ({e}). Instale kaleido: pip install kaleido")

Export PNG falhou (
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
). Instale kaleido: pip install kaleido


---

## Insight 1 — Nota média por UF

Fonte: `agregado_municipio_2022.csv` (já no pipeline — sem reprocessar o CSV bruto).  
**Média ponderada** por número de participantes de cada município → evita que cidades pequenas distorçam a média estadual.

In [53]:
import numpy as np

# Colunas confirmadas: SG_UF_PROVA, TOTAL_PARTICIPANTES, MEDIA_GERAL
df_muni = pd.read_csv(PASTA_ANALYTICS / 'agregado_municipio_2022.csv', sep=';', encoding='utf-8')
print("Colunas:", list(df_muni.columns))

# Média ponderada município → UF
df_muni['_SOMA'] = df_muni['MEDIA_GERAL'] * df_muni['TOTAL_PARTICIPANTES']
df_uf = (
    df_muni
    .groupby('SG_UF_PROVA', as_index=False)
    .agg(_SOMA=('_SOMA', 'sum'), TOTAL_PARTICIPANTES=('TOTAL_PARTICIPANTES', 'sum'))
)
df_uf['MEDIA_GERAL'] = df_uf['_SOMA'] / df_uf['TOTAL_PARTICIPANTES']
df_uf = df_uf.drop(columns='_SOMA').sort_values('MEDIA_GERAL', ascending=False).reset_index(drop=True)

mapa_uf = dict(zip(df_uf['SG_UF_PROVA'], df_uf['MEDIA_GERAL'].round(1)))

print(f"\nUFs: {len(df_uf)} | N total: {df_uf['TOTAL_PARTICIPANTES'].sum():,}".replace(',', '.'))
print("\nTop 5:")
print(df_uf.head(5)[['SG_UF_PROVA', 'MEDIA_GERAL', 'TOTAL_PARTICIPANTES']].to_string(index=False))
print("\nBottom 5:")
print(df_uf.tail(5)[['SG_UF_PROVA', 'MEDIA_GERAL', 'TOTAL_PARTICIPANTES']].to_string(index=False))

Colunas: ['CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA', 'SG_UF_PROVA', 'PORTE_MUNICIPIO', 'TOTAL_PARTICIPANTES', 'MEDIA_GERAL', 'MEDIA_REDACAO', 'MEDIA_OBJETIVAS', 'POPULACAO_MUNICIPIO']

UFs: 27 | N total: 2.504.014

Top 5:
SG_UF_PROVA  MEDIA_GERAL  TOTAL_PARTICIPANTES
         MG   562.544531               226971
         SP   561.764291               398601
         SC   556.500625                60393
         DF   555.713560                47566
         RJ   553.087101               180402

Bottom 5:
SG_UF_PROVA  MEDIA_GERAL  TOTAL_PARTICIPANTES
         AC   511.438377                15486
         PA   509.704222               143867
         MA   508.037705                95899
         AP   504.959003                15544
         AM   496.462089                46307


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

COR_TEAL  = '#18BC9C'
COR_NAVY  = '#2C3E50'
COR_CINZA = '#8FA3B0'

# Mapeamento UF → código IBGE numérico (2 dígitos, string) para o GeoJSON da malha IBGE
UF_CODIGOS = {
    'AC':'12','AL':'27','AM':'13','AP':'16','BA':'29','CE':'23','DF':'53',
    'ES':'32','GO':'52','MA':'21','MG':'31','MS':'50','MT':'51','PA':'15',
    'PB':'25','PE':'26','PI':'22','PR':'41','RJ':'33','RN':'24','RO':'11',
    'RR':'14','RS':'43','SC':'42','SE':'28','SP':'35','TO':'17',
}
df_uf['CODAREA'] = df_uf['SG_UF_PROVA'].map(UF_CODIGOS)

geojson_ok = False
try:
    import requests
    resp = requests.get(
        'https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR'
        '?formato=application/vnd.geo+json&qualidade=intermediaria&intrarregiao=UF',
        timeout=15,
    )
    if resp.status_code == 200:
        geojson_br = resp.json()
        feat0   = geojson_br['features'][0]
        feat_id = feat0.get('id')
        feat_pr = feat0.get('properties', {})
        print(f"GeoJSON IBGE OK. id exemplo: {feat_id!r} | properties: {feat_pr}")
        # IBGE coloca o código numérico em 'id'; se vier como int, converte para str
        if isinstance(feat_id, int):
            for f in geojson_br['features']:
                f['id'] = str(f['id'])
        fkey = 'id'
        geojson_ok = True
    else:
        print(f"IBGE retornou {resp.status_code} — usando fallback barras.")
except Exception as e:
    print(f"GeoJSON indisponível ({e}) — usando fallback barras.")

if geojson_ok:
    fig_mapa = px.choropleth(
        df_uf,
        geojson=geojson_br,
        locations='CODAREA',
        featureidkey='properties.codarea',
        color='MEDIA_GERAL',
        color_continuous_scale=[[0, COR_CINZA], [0.5, COR_TEAL], [1, COR_NAVY]],
        hover_name='SG_UF_PROVA',
        hover_data={'TOTAL_PARTICIPANTES': True, 'MEDIA_GERAL': ':.1f', 'CODAREA': False},
        labels={'MEDIA_GERAL': 'Média (pts)'},
        fitbounds='locations',
        basemap_visible=False,
    )
    fig_mapa.update_layout(
        title='Nota média por UF — ENEM 2022',
        paper_bgcolor='white', font=dict(color=COR_NAVY, family='Arial'),
        coloraxis_colorbar_title='Pts',
        margin=dict(l=0, r=0, t=50, b=0), width=750, height=550,
    )
else:
    # Fallback: barras horizontais ranqueadas por UF
    df_bar = df_uf.sort_values('MEDIA_GERAL')
    fig_mapa = go.Figure(go.Bar(
        x=df_bar['MEDIA_GERAL'],
        y=df_bar['SG_UF_PROVA'],
        orientation='h',
        marker=dict(
            color=df_bar['MEDIA_GERAL'],
            colorscale=[[0, COR_CINZA], [0.5, COR_TEAL], [1, COR_NAVY]],
            showscale=True, colorbar=dict(title='Pts'),
        ),
        text=df_bar['MEDIA_GERAL'].round(1).astype(str),
        textposition='outside', textfont=dict(size=9),
    ))
    fig_mapa.update_layout(
        title='Nota média por UF — ENEM 2022 (fallback barras)',
        xaxis=dict(range=[490, 595], showgrid=False, showticklabels=False),
        yaxis=dict(tickfont=dict(size=10)),
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(color=COR_NAVY, family='Arial'),
        height=700, margin=dict(l=50, r=80, t=50, b=20),
    )

fig_mapa.show()

saida_mapa = Path.cwd() / 'mapa_uf.png'
try:
    fig_mapa.write_image(str(saida_mapa), scale=2)
    print(f"PNG salvo: {saida_mapa}")
except Exception as e:
    print(f"Export PNG falhou: {e}. pip install kaleido")

GeoJSON IBGE OK. id exemplo: None | properties: {'codarea': '11'}


NameError: name 'properties' is not defined

---

## Insight 2 — Ranking de fatores que mais explicam a nota

**Modelo**: RandomForestRegressor + **importância por permutação** (mede a queda de R² quando a coluna é embaralhada — mais honesta que `feature_importances_` nativa do RF).  
**Ressalva obrigatória**: os valores medem *associação estatística*, não causalidade. Um fator pode ser proxy de outro (ex.: escola privada e renda são correlacionados).

**Features disponíveis no pipeline** (colunas confirmadas em `enem_2022_ibge_transformado.csv`):  
`RENDA_FAMILIAR` (ordinal 0–16), `ACESSO_INTERNET`, `TIPO_ESCOLA`, `TP_SEXO`, `TP_FAIXA_ETARIA`, `PORTE_MUNICIPIO`.

**Ausentes** (não foram incluídas em `COLUNAS_DESEJADAS` em `data_processor.py`):  
`TP_COR_RACA`, `Q001/Q002` (escolaridade dos pais), `Q024` (computador), `NU_IDADE`.

In [55]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

ORDEM_RENDA_FULL = [
    "Nenhuma renda", "Até 1 salário mínimo", "1 a 1,5 salários", "1,5 a 2 salários",
    "2 a 2,5 salários", "2,5 a 3 salários", "3 a 4 salários", "4 a 5 salários",
    "5 a 6 salários", "6 a 7 salários", "7 a 8 salários", "8 a 9 salários",
    "9 a 10 salários", "10 a 12 salários", "12 a 15 salários", "15 a 20 salários",
    "Acima de 20 salários",
]
renda_map = {v: i for i, v in enumerate(ORDEM_RENDA_FULL)}  # "Nenhuma renda"=0 ... "Acima de 20"=16

ORDEM_PORTE_FULL = [
    "Até 20 mil", "20 mil a 100 mil", "100 mil a 500 mil",
    "500 mil a 1 milhão", "Acima de 1 milhão",
]
porte_map = {v: i for i, v in enumerate(ORDEM_PORTE_FULL)}  # 0-4

# Lê apenas as 7 colunas necessárias — evita carregar 2,5M × 30 col na íntegra
print("Carregando colunas selecionadas do dataset transformado...")
_cols = ['MEDIA_GERAL', 'RENDA_FAMILIAR', 'ACESSO_INTERNET',
         'TIPO_ESCOLA', 'TP_SEXO', 'TP_FAIXA_ETARIA', 'PORTE_MUNICIPIO']
df_full_sel = pd.read_csv(
    PASTA_ANALYTICS / 'enem_2022_ibge_transformado.csv',
    sep=';', encoding='utf-8', usecols=_cols,
)
df_sample = (
    df_full_sel
    .dropna(subset=['MEDIA_GERAL'])
    .sample(100_000, random_state=42)
    .reset_index(drop=True)
)
del df_full_sel
print(f"Amostra: {len(df_sample):,} linhas".replace(',', '.'))

# Encoding determinístico (sem one-hot verbose): 7 features limpas
df_feat = pd.DataFrame({
    'Renda familiar'   : df_sample['RENDA_FAMILIAR'].map(renda_map).fillna(-1),
    'Escola privada'   : (df_sample['TIPO_ESCOLA'] == 'Privada').astype(float),
    'Escola pública'   : (df_sample['TIPO_ESCOLA'] == 'Pública').astype(float),
    'Internet em casa' : (df_sample['ACESSO_INTERNET'] == 'Sim').astype(float),
    'Porte município'  : df_sample['PORTE_MUNICIPIO'].map(porte_map).fillna(-1),
    'Faixa etária'     : pd.to_numeric(df_sample['TP_FAIXA_ETARIA'], errors='coerce').fillna(-1),
    'Sexo masculino'   : (df_sample['TP_SEXO'] == 'M').astype(float),
})
FEATURE_COLS = list(df_feat.columns)
X = df_feat.values
y = df_sample['MEDIA_GERAL'].values

print("Features:", FEATURE_COLS)
print(f"Nulos em X: {np.isnan(X).sum()} | shape: {X.shape}")

Carregando colunas selecionadas do dataset transformado...
Amostra: 100.000 linhas
Features: ['Renda familiar', 'Escola privada', 'Escola pública', 'Internet em casa', 'Porte município', 'Faixa etária', 'Sexo masculino']
Nulos em X: 0 | shape: (100000, 7)


In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Treinando RandomForest (100 árvores, max_depth=10)…")
rf = RandomForestRegressor(
    n_estimators=100, max_depth=10, min_samples_leaf=50,
    n_jobs=-1, random_state=42,
)
rf.fit(X_train, y_train)
r2_rf = r2_score(y_test, rf.predict(X_test))
print(f"R² no conjunto de teste: {r2_rf:.4f}")

print("Calculando importância por permutação (5 repetições no conjunto de teste)…")
perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)

df_imp = (
    pd.DataFrame({
        'feature'    : FEATURE_COLS,
        'imp_mean'   : perm.importances_mean,
        'imp_std'    : perm.importances_std,
    })
    .sort_values('imp_mean', ascending=False)
    .reset_index(drop=True)
)
print("\nRanking (queda média de R² por permutação):")
print(df_imp.to_string(index=False))

# Gráfico barras horizontais
df_plot = df_imp.sort_values('imp_mean')  # ascending → mais importante no topo
cores = [COR_TEAL if f == 'Renda familiar'
         else COR_NAVY if 'Escola' in f
         else COR_CINZA
         for f in df_plot['feature']]

fig_rank = go.Figure(go.Bar(
    x=df_plot['imp_mean'],
    y=df_plot['feature'],
    orientation='h',
    error_x=dict(type='data', array=df_plot['imp_std'].values, visible=True, color=COR_CINZA),
    marker_color=cores,
    text=df_plot['imp_mean'].map(lambda v: f"{v:.4f}"),
    textposition='outside',
    textfont=dict(size=10, color=COR_NAVY),
    cliponaxis=False,
))
fig_rank.update_layout(
    title=f'Ranking de fatores — importância por permutação | R² teste = {r2_rf:.3f}',
    xaxis=dict(title='Queda média de R² (quanto maior, mais importante)', showgrid=False, showticklabels=False),
    yaxis=dict(tickfont=dict(size=12, color=COR_NAVY)),
    paper_bgcolor='white', plot_bgcolor='white',
    font=dict(color=COR_NAVY, family='Arial'),
    height=380, margin=dict(l=140, r=90, t=55, b=40),
)
fig_rank.show()

ranking_fatores = dict(zip(df_imp['feature'], df_imp['imp_mean'].round(5)))
print("\nranking_fatores =", ranking_fatores)

saida_rank = Path.cwd() / 'ranking_fatores.png'
try:
    fig_rank.write_image(str(saida_rank), scale=2, width=850, height=480)
    print(f"PNG salvo: {saida_rank}")
except Exception as e:
    print(f"Export PNG falhou: {e}. pip install kaleido")

Treinando RandomForest (100 árvores, max_depth=10)…
R² no conjunto de teste: 0.2580
Calculando importância por permutação (5 repetições no conjunto de teste)…

Ranking (queda média de R² por permutação):
         feature  imp_mean  imp_std
  Renda familiar  0.334911 0.005572
  Escola pública  0.079756 0.001396
    Faixa etária  0.073981 0.003486
  Escola privada  0.023310 0.000939
 Porte município  0.007515 0.000784
Internet em casa  0.006122 0.000656
  Sexo masculino  0.002335 0.000538



ranking_fatores = {'Renda familiar': 0.33491, 'Escola pública': 0.07976, 'Faixa etária': 0.07398, 'Escola privada': 0.02331, 'Porte município': 0.00752, 'Internet em casa': 0.00612, 'Sexo masculino': 0.00233}
Export PNG falhou: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
. pip install kaleido


---

## Insight 3 — Gap escola pública vs. privada controlando por renda

> **Substituição aprovada:** `TP_COR_RACA` não foi incluída em `COLUNAS_DESEJADAS` no `data_processor.py`
> e portanto não existe em nenhum CSV do pipeline. O insight racial original exigiria reprocessar
> os microdados brutos. Em substituição: **gap por tipo de escola controlado por renda** —
> responde "quanto da vantagem da escola privada é atribuível à escola em si, e não à renda do aluno?".

**Parte A** — heatmap descritivo: nota média por escola × faixa de renda.  
**Parte B** — regressão linear `nota ~ renda_ordinal + privada` para isolar o coeficiente da escola.

In [57]:
# Parte A — heatmap descritivo
# Reutiliza df_sample (100k linhas já carregadas no Insight 2)
df_er = (
    df_sample
    .dropna(subset=['TIPO_ESCOLA', 'RENDA_FAMILIAR', 'MEDIA_GERAL'])
    .query("TIPO_ESCOLA != 'Não respondeu'")
    .copy()
)
df_er['RENDA_ORD'] = df_er['RENDA_FAMILIAR'].map(renda_map)

print(f"N após filtro (Privada/Pública com renda válida): {len(df_er):,}".replace(',', '.'))
print(df_er['TIPO_ESCOLA'].value_counts().to_string())

# Agrega escola × renda
df_grupo = (
    df_er
    .groupby(['TIPO_ESCOLA', 'RENDA_FAMILIAR'], as_index=False)
    .agg(MEDIA_GERAL=('MEDIA_GERAL', 'mean'), N=('MEDIA_GERAL', 'count'))
)
df_grupo['RENDA_ORD'] = df_grupo['RENDA_FAMILIAR'].map(renda_map)
df_grupo = df_grupo.sort_values('RENDA_ORD')

# Pivot para heatmap
pivot = df_grupo.pivot(index='TIPO_ESCOLA', columns='RENDA_FAMILIAR', values='MEDIA_GERAL')
cols_ord = [r for r in ORDEM_RENDA_FULL if r in pivot.columns]
pivot = pivot[cols_ord]

# Rótulos curtos para eixo X
RENDA_CURTA = {
    "Nenhuma renda": "Nenhuma", "Até 1 salário mínimo": "≤1",
    "1 a 1,5 salários": "1–1,5", "1,5 a 2 salários": "1,5–2",
    "2 a 2,5 salários": "2–2,5", "2,5 a 3 salários": "2,5–3",
    "3 a 4 salários": "3–4", "4 a 5 salários": "4–5",
    "5 a 6 salários": "5–6", "6 a 7 salários": "6–7",
    "7 a 8 salários": "7–8", "8 a 9 salários": "8–9",
    "9 a 10 salários": "9–10", "10 a 12 salários": "10–12",
    "12 a 15 salários": "12–15", "15 a 20 salários": "15–20",
    "Acima de 20 salários": ">20",
}
x_labels = [RENDA_CURTA.get(c, c) for c in cols_ord]

fig_heat = go.Figure(go.Heatmap(
    z=pivot.values,
    x=x_labels,
    y=pivot.index.tolist(),
    colorscale=[[0, '#8FA3B0'], [0.5, '#18BC9C'], [1, '#2C3E50']],
    text=[[f'{v:.0f}' if not np.isnan(v) else '' for v in row] for row in pivot.values],
    texttemplate='%{text}',
    hovertemplate='Renda: %{x}<br>Escola: %{y}<br>Nota: %{z:.1f}<extra></extra>',
    colorbar=dict(title='Nota (pts)'),
))
fig_heat.update_layout(
    title='Nota média: tipo de escola × faixa de renda — ENEM 2022',
    xaxis=dict(title='Renda familiar (salários mínimos)', tickfont=dict(size=9), tickangle=-45),
    yaxis=dict(title=''),
    paper_bgcolor='white', plot_bgcolor='white',
    font=dict(color=COR_NAVY, family='Arial'),
    height=300, margin=dict(l=80, r=30, t=55, b=90),
)
fig_heat.show()

saida_heat = Path.cwd() / 'gap_escola_renda.png'
try:
    fig_heat.write_image(str(saida_heat), scale=2, width=1050, height=420)
    print(f"PNG salvo: {saida_heat}")
except Exception as e:
    print(f"Export PNG falhou: {e}. pip install kaleido")

N após filtro (Privada/Pública com renda válida): 40.777
TIPO_ESCOLA
Pública    32813
Privada     7964


Export PNG falhou: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido
. pip install kaleido


In [58]:
# Parte B — regressão linear: nota ~ renda_ordinal + escola_privada
from sklearn.linear_model import LinearRegression

df_reg = df_er.dropna(subset=['RENDA_ORD']).copy()
df_reg['PRIVADA'] = (df_reg['TIPO_ESCOLA'] == 'Privada').astype(float)

X_reg = df_reg[['RENDA_ORD', 'PRIVADA']].values
y_reg = df_reg['MEDIA_GERAL'].values

lr = LinearRegression().fit(X_reg, y_reg)
r2_escola = r2_score(y_reg, lr.predict(X_reg))

coef_renda_pts   = lr.coef_[0]   # ganho por faixa de renda (mantendo escola constante)
coef_privada_pts = lr.coef_[1]   # gap privada vs pública JÁ controlado por renda

# Gap bruto (diferença simples de médias, sem controle)
media_priv_amostra = df_reg[df_reg['TIPO_ESCOLA'] == 'Privada']['MEDIA_GERAL'].mean()
media_pub_amostra  = df_reg[df_reg['TIPO_ESCOLA'] == 'Pública' ]['MEDIA_GERAL'].mean()
gap_bruto = media_priv_amostra - media_pub_amostra

def _br(v): return f"{v:.1f}".replace('.', ',')

print("=" * 55)
print("  Escola Privada vs Pública — ENEM 2022")
print("=" * 55)
print(f"  Gap bruto (simples):         {gap_bruto:+.1f} pts")
print(f"  Gap ajustado (ctrl. renda):  {coef_privada_pts:+.1f} pts")
print(f"  Ganho por faixa de renda:    {coef_renda_pts:+.2f} pts/faixa")
print(f"  R² modelo linear:            {r2_escola:.4f}")
print(f"  n_privada (amostra): {(df_reg['TIPO_ESCOLA']=='Privada').sum():,}  |  "
      f"n_pública: {(df_reg['TIPO_ESCOLA']=='Pública').sum():,}".replace(',', '.'))
print("=" * 55)
print()
print(f"  → Mesmo na MESMA faixa de renda, alunos de escola")
print(f"    privada tiram ~{_br(coef_privada_pts)} pts a mais que alunos de")
print(f"    escola pública. O gap bruto de {_br(gap_bruto)} pts é explicado")
print(f"    parcialmente pela renda, mas não integralmente.")

gap_escola_renda = {
    'gap_bruto_privada_vs_publica'   : round(gap_bruto,         1),
    'gap_ajustado_privada_vs_publica': round(coef_privada_pts,  1),
    'ganho_por_faixa_renda'          : round(coef_renda_pts,    2),
    'r2_regressao_linear'            : round(r2_escola,         4),
    'n_privada_amostra'              : int((df_reg['TIPO_ESCOLA']=='Privada').sum()),
    'n_publica_amostra'              : int((df_reg['TIPO_ESCOLA']=='Pública' ).sum()),
}
print("\ngap_escola_renda =", gap_escola_renda)

  Escola Privada vs Pública — ENEM 2022
  Gap bruto (simples):         +92.1 pts
  Gap ajustado (ctrl. renda):  +48.8 pts
  Ganho por faixa de renda:    +8.30 pts/faixa
  R² modelo linear:            0.2362
  n_privada (amostra): 7.964  |  n_pública: 32.813

  → Mesmo na MESMA faixa de renda, alunos de escola
    privada tiram ~48,8 pts a mais que alunos de
    escola pública. O gap bruto de 92,1 pts é explicado
    parcialmente pela renda, mas não integralmente.

gap_escola_renda = {'gap_bruto_privada_vs_publica': np.float64(92.1), 'gap_ajustado_privada_vs_publica': np.float64(48.8), 'ganho_por_faixa_renda': np.float64(8.3), 'r2_regressao_linear': 0.2362, 'n_privada_amostra': 7964, 'n_publica_amostra': 32813}


---

## Resumo Consolidado

Todos os números dos 4 blocos do notebook — prontos para copiar na apresentação.

In [59]:
sep = "=" * 60

print(sep)
print("  BLOCO 1 — TABELA COMPARATIVA (N = 2.504.014)")
print(sep)
print(f"  Alta Renda (faixa Q)  : {_br(media_alta_renda)} pts  ({dif_alta_renda:+.1f}% vs. nac.)")
print(f"  Escola Privada        : {_br(media_privada)} pts  ({dif_privada:+.1f}% vs. nac.)")
print(f"  Com Internet          : {_br(media_internet)} pts  ({dif_internet:+.1f}% vs. nac.)")
print(f"  Média Nacional        : {_br(media_nacional)} pts  (Ref. 0%)")

print()
print(sep)
print("  BLOCO 2 — BARREIRA DIGITAL")
print(sep)
print(f"  Com Internet  : {_br(barreira_digital['media_com_internet'])} pts  (n = {barreira_digital['n_com_internet']:,})".replace(',', '.'))
print(f"  Sem Internet  : {_br(barreira_digital['media_sem_internet'])} pts  (n = {barreira_digital['n_sem_internet']:,})".replace(',', '.'))
print(f"  Penalidade    : ~{_br(barreira_digital['penalidade_absoluta'])} pontos  (−{_br(barreira_digital['penalidade_relativa_pct'])}%)")
print(f"  % sem internet: {_br(barreira_digital['pct_sem_internet'])}% da base")

print()
print(sep)
print("  INSIGHT 1 — MAPA UF (top 5 / bottom 5)")
print(sep)
top5    = df_uf.head(5)[['SG_UF_PROVA', 'MEDIA_GERAL']].values
bottom5 = df_uf.tail(5)[['SG_UF_PROVA', 'MEDIA_GERAL']].values
print("  Top 5:   ", "  ".join(f"{uf} {_br(m)}" for uf, m in top5))
print("  Bottom 5:", "  ".join(f"{uf} {_br(m)}" for uf, m in bottom5))

print()
print(sep)
print(f"  INSIGHT 2 — RANKING FATORES (R² RF = {r2_rf:.3f})")
print(sep)
for i, (feat, imp) in enumerate(ranking_fatores.items(), 1):
    print(f"  {i}. {feat:<20} imp = {imp:.5f}")

print()
print(sep)
print("  INSIGHT 3 — GAP ESCOLA (amostra 100k, random_state=42)")
print(sep)
print(f"  Gap bruto (Priv. − Públ.) : {gap_escola_renda['gap_bruto_privada_vs_publica']:+.1f} pts")
print(f"  Gap ajustado (ctrl. renda): {gap_escola_renda['gap_ajustado_privada_vs_publica']:+.1f} pts")
print(f"  Ganho por faixa de renda  : {gap_escola_renda['ganho_por_faixa_renda']:+.2f} pts/faixa")
print(f"  R² regressão linear       : {gap_escola_renda['r2_regressao_linear']:.4f}")

print()
print(sep)
print("  PREMISSAS E LIMITAÇÕES")
print(sep)
premissas = [
    "MEDIA_GERAL = mean(CN, CH, LC, MT, Redação) com mean(axis=1) — candidatos com nota",
    "  parcial têm média calculada sobre provas disponíveis (não foram excluídos).",
    "Candidatos com TODAS as notas nulas foram removidos em data_processor.py (dropna how=all).",
    "Bloco 1: mapa UF usa média PONDERADA por participantes de cada município.",
    "Insight 2: amostra aleatória de 100k linhas (random_state=42) do dataset de 2,5M.",
    "  Features ausentes no pipeline (TP_COR_RACA, Q001/Q002, Q024, NU_IDADE) não foram",
    "  incluídas — o R² real (com mais features) seria maior.",
    "  Importância por permutação mede associação, NÃO causalidade.",
    "Insight 3: amostra = mesma 100k. Exclui 'Não respondeu' (~60% dos candidatos).",
    "  Regressão linear simples (2 preditores); omite confundidores não disponíveis.",
    "  Gap ajustado é uma estimativa de limite inferior do efeito escola.",
    "Ano-base: ENEM 2022. N total = 2.504.014 participantes.",
]
for p in premissas:
    print(f"  • {p}" if not p.startswith("  ") else f"   {p}")

  BLOCO 1 — TABELA COMPARATIVA (N = 2.504.014)
  Alta Renda (faixa Q)  : 635,3 pts  (+17.9% vs. nac.)
  Escola Privada        : 606,7 pts  (+12.6% vs. nac.)
  Com Internet          : 544,0 pts  (+0.9% vs. nac.)
  Média Nacional        : 539,0 pts  (Ref. 0%)

  BLOCO 2 — BARREIRA DIGITAL
  Com Internet  : 544.0 pts  (n = 2.298.919)
  Sem Internet  : 482.2 pts  (n = 205.095)
  Penalidade    : ~61,9 pontos  (−11,4%)
  % sem internet: 8,2% da base

  INSIGHT 1 — MAPA UF (top 5 / bottom 5)
  Top 5:    MG 562,5  SP 561,8  SC 556,5  DF 555,7  RJ 553,1
  Bottom 5: AC 511,4  PA 509,7  MA 508,0  AP 505,0  AM 496,5

  INSIGHT 2 — RANKING FATORES (R² RF = 0.258)
  1. Renda familiar       imp = 0.33491
  2. Escola pública       imp = 0.07976
  3. Faixa etária         imp = 0.07398
  4. Escola privada       imp = 0.02331
  5. Porte município      imp = 0.00752
  6. Internet em casa     imp = 0.00612
  7. Sexo masculino       imp = 0.00233

  INSIGHT 3 — GAP ESCOLA (amostra 100k, random_state=42)
  G